In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = 'musa5090s26-team5'
client = bigquery.Client(project=PROJECT_ID)

print("BigQuery client ready.")

BigQuery client ready.


In [3]:
query = """
SELECT *
FROM `musa5090s26-team5.core.opa_assessments`
LIMIT 3
"""
df_test = client.query(query).to_dataframe(create_bqstorage_client=False)
print(df_test.columns.tolist())
df_test.head()

['property_id', 'parcel_number', 'year', 'market_value', 'taxable_land', 'taxable_building', 'exempt_land', 'exempt_building', 'objectid']


,property_id,parcel_number,year,market_value,taxable_land,taxable_building,exempt_land,exempt_building,objectid
0,21013212,21013212,2016.0,0.0,0.0,0.0,0.0,0.0,90786
1,23225605,23225605,2017.0,0.0,0.0,0.0,0.0,0.0,178862
2,51173130,51173130,2018.0,0.0,0.0,0.0,0.0,0.0,363509


In [4]:
# Step 1: OPA Assessments - 历史评估趋势
query = """
SELECT
    parcel_number,
    year,
    market_value
FROM `musa5090s26-team5.core.opa_assessments`
WHERE market_value IS NOT NULL
    AND market_value > 0
ORDER BY parcel_number, year
"""

assessments = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"Rows: {len(assessments):,}")
assessments.head()

Rows: 6,890,147


,parcel_number,year,market_value
0,11000017,2023.0,123000.0
1,11000017,2024.0,670000.0
2,11000017,2025.0,594000.0
3,11000017,2026.0,594000.0
4,11000018,2023.0,125000.0


In [5]:
# Step 1b: 计算年度变化率特征
query = """
WITH yearly AS (
    SELECT
        parcel_number,
        year,
        market_value,
        LAG(market_value) OVER (PARTITION BY parcel_number ORDER BY year) AS prev_year_value
    FROM `musa5090s26-team5.core.opa_assessments`
    WHERE market_value IS NOT NULL
        AND market_value > 0
)
SELECT
    parcel_number,
    MAX(CASE WHEN year = 2023 THEN market_value END) AS assessed_value_2023,
    MAX(CASE WHEN year = 2024 THEN market_value END) AS assessed_value_2024,
    MAX(CASE WHEN year = 2025 THEN market_value END) AS assessed_value_2025,
    ROUND(
        SAFE_DIVIDE(
            MAX(CASE WHEN year = 2025 THEN market_value END) -
            MAX(CASE WHEN year = 2023 THEN market_value END),
            MAX(CASE WHEN year = 2023 THEN market_value END)
        ) * 100, 2
    ) AS pct_change_2023_to_2025
FROM yearly
GROUP BY parcel_number
"""

assessments_features = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"Rows: {len(assessments_features):,}")
assessments_features.head(10)

Rows: 583,425


,parcel_number,assessed_value_2023,assessed_value_2024,assessed_value_2025,pct_change_2023_to_2025
0,192321010,100.0,100.0,33800.0,33700.00
1,885181720,400.0,400.0,1200.0,200.00
2,182240100,87300.0,87300.0,88800.0,1.72
3,581128301,12500.0,12500.0,40000.0,220.00
4,331217400,41400.0,41400.0,8800.0,-78.74
5,885312880,600.0,600.0,1000.0,66.67
6,372452000,7900.0,7900.0,7500.0,-5.06
7,885576540,1500.0,1500.0,3100.0,106.67
8,463202001,800.0,800.0,13800.0,1625.00
9,183189910,48100.0,48100.0,64600.0,34.30


In [7]:
# Step 2: PWD Parcels - check column names first
query = """
SELECT *
FROM `musa5090s26-team5.core.pwd_parcels`
LIMIT 3
"""
pwd_test = client.query(query).to_dataframe(create_bqstorage_client=False)
print(pwd_test.columns.tolist())
pwd_test.head()

['property_id', 'objectid', 'parcelid', 'tencode', 'address', 'owner1', 'owner2', 'bldg_code', 'bldg_desc', 'brt_id', 'num_brt', 'num_accounts', 'gross_area', 'pin', 'parcel_id', 'shape__area', 'shape__length', 'geometry']


,property_id,objectid,parcelid,tencode,address,owner1,owner2,bldg_code,bldg_desc,brt_id,num_brt,num_accounts,gross_area,pin,parcel_id,shape__area,shape__length,geometry
0,None,400,1475441,1622007529,7527 BATTERSBY ST,LIN FENG,CHEN YAN YUN,None,None,None,1.0,1,4144,NaN,1475441,657.789062,115.725477,"POLYGON((-75.0470687585718 40.0453871613238, -..."
1,432123700,43910,47079,8799003649,3649-53 N 10TH ST,RAMOS ERNEST,RAMOS GUILLERMINA,None,None,432123700,1.0,1,47030,NaN,47079,7456.320312,366.738490,"POLYGON((-75.1437597104499 40.0078690476003, -..."
2,073209300,80218,86175,1204000305,305 E ALLEGHENY AVE,GUINEA REAL ESTATE INVESTMENTS LLC,,None,None,073209300,1.0,1,5378,NaN,86175,852.527344,116.806081,"POLYGON((-75.1248646730003 39.9983440681416, -..."


In [8]:
# Step 2: Extract spatial features from PWD Parcels
query = """
SELECT
    brt_id AS parcel_number,
    gross_area,
    shape__area AS lot_area_sqft,
    shape__length AS lot_perimeter,
    ROUND(SAFE_DIVIDE(shape__area, shape__length * shape__length), 4) AS lot_shape_ratio
FROM `musa5090s26-team5.core.pwd_parcels`
WHERE brt_id IS NOT NULL
    AND shape__area IS NOT NULL
    AND shape__area > 0
"""

pwd_features = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"Rows: {len(pwd_features):,}")
pwd_features.head(10)

Rows: 545,847


,parcel_number,gross_area,lot_area_sqft,lot_perimeter,lot_shape_ratio
0,432123700,47030,7456.320312,366.738490,0.0554
1,073209300,5378,852.527344,116.806081,0.0625
2,394456610,2306,364.808594,106.019540,0.0325
3,151040810,2140,338.910156,84.428501,0.0475
4,885046340,2389,378.292969,85.399446,0.0519
5,482127105,669,105.957031,49.170303,0.0438
6,212289848,4996,792.804688,250.499668,0.0126
7,183269500,941,149.195312,55.500475,0.0484
8,885317680,5410,857.597656,120.101803,0.0595
9,453205400,2836,449.605469,88.281568,0.0577


In [11]:
# Step 3: Download Philadelphia neighborhoods using requests
import geopandas as gpd
import requests
from io import BytesIO

url = "https://services.arcgis.com/fLeGjb7u4uXqeF9q/arcgis/rest/services/Philadelphia_Neighborhoods/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"

response = requests.get(url)
neighborhoods = gpd.read_file(BytesIO(response.content))
print(f"Rows: {len(neighborhoods)}")
print(neighborhoods.columns.tolist())
neighborhoods.head()

Rows: 57
['id', 'ObjectId', 'Shape__Area', 'Shape__Length', 'geometry']


,id,ObjectId,Shape__Area,Shape__Length,geometry
0,Allegheny West,1,7.148906e+06,13391.105904,"POLYGON ((-75.16242 40.00282, -75.16084 40.002..."
1,Bella Vista/Southwark,2,3.721612e+06,8209.752810,"POLYGON ((-75.15573 39.92674, -75.15582 39.926..."
2,Bridesburg,3,7.259312e+06,11411.057478,"POLYGON ((-75.06835 40.00692, -75.0677 40.0053..."
3,Bustleton,4,2.182823e+07,22329.414488,"POLYGON ((-75.03635 40.12017, -75.0352 40.1194..."
4,Cedarbrook/Stenton,5,8.192725e+06,11773.792446,"POLYGON ((-75.16014 40.0574, -75.16069 40.0568..."


In [13]:
# Check opa_properties column names
query = """
SELECT *
FROM `musa5090s26-team5.core.opa_properties`
LIMIT 2
"""
opa_test = client.query(query).to_dataframe(create_bqstorage_client=False)
print(opa_test.columns.tolist())

['property_id', 'assessment_date', 'basements', 'beginning_point', 'book_and_page', 'building_code', 'building_code_description', 'category_code', 'category_code_description', 'census_tract', 'central_air', 'cross_reference', 'date_exterior_condition', 'depth', 'exempt_building', 'exempt_land', 'exterior_condition', 'fireplaces', 'frontage', 'fuel', 'garage_spaces', 'garage_type', 'general_construction', 'geographic_ward', 'homestead_exemption', 'house_extension', 'house_number', 'interior_condition', 'location', 'mailing_address_1', 'mailing_address_2', 'mailing_care_of', 'mailing_city_state', 'mailing_street', 'mailing_zip', 'market_value', 'market_value_date', 'number_of_bathrooms', 'number_of_bedrooms', 'number_of_rooms', 'number_stories', 'off_street_open', 'other_building', 'owner_1', 'owner_2', 'parcel_number', 'parcel_shape', 'quality_grade', 'recording_date', 'registry_number', 'sale_date', 'sale_price', 'separate_utilities', 'sewer', 'site_type', 'state_code', 'street_code', 

In [14]:
# Step 3b: Pull OPA properties with geometry for spatial join
query = """
SELECT
    parcel_number,
    zip_code,
    ST_AsText(geometry) AS geom_wkt
FROM `musa5090s26-team5.core.opa_properties`
WHERE geometry IS NOT NULL
"""
opa_geo = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"Rows: {len(opa_geo):,}")
opa_geo.head()

Rows: 583,365


,parcel_number,zip_code,geom_wkt
0,788043500,19112,POINT(-75.167387 39.891625)
1,871000023,19122,POINT(-75.137748 39.980007)
2,783000004,19131,POINT(-75.229957 39.969639)
3,871000071,19133,POINT(-75.136692 39.984021)
4,132000002,19140,POINT(-75.153567 40.022044)


In [15]:
# Step 3c: Spatial join OPA properties to neighborhoods
from shapely.wkt import loads

# Convert WKT to geometry
opa_geo['geometry'] = opa_geo['geom_wkt'].apply(loads)
opa_gdf = gpd.GeoDataFrame(opa_geo, geometry='geometry', crs='EPSG:4326')

# Make sure neighborhoods CRS matches
neighborhoods = neighborhoods.to_crs('EPSG:4326')

# Spatial join
opa_with_neighborhoods = gpd.sjoin(
    opa_gdf[['parcel_number', 'zip_code', 'geometry']],
    neighborhoods[['id', 'geometry']],
    how='left',
    predicate='within'
)

opa_with_neighborhoods = opa_with_neighborhoods[['parcel_number', 'zip_code', 'id']].rename(columns={'id': 'neighborhood'})

print(f"Rows: {len(opa_with_neighborhoods):,}")
print(f"Null neighborhoods: {opa_with_neighborhoods['neighborhood'].isna().sum():,}")
opa_with_neighborhoods.head(10)

Rows: 583,365
Null neighborhoods: 4


,parcel_number,zip_code,neighborhood
0,788043500,19112,Industrial
1,871000023,19122,North Philadelphia/East
2,783000004,19131,West Philadelphia/Parkside
3,871000071,19133,North Philadelphia/East
4,132000002,19140,Tioga/Nicetown
5,871601640,19134,Kensington
6,874500512,19145,South Philadelphia/West
7,871000020,19122,North Philadelphia/East
8,871508657,19107,North Philadelphia/East
9,871402950,19120,Olney


In [22]:
# Step 4: SEPTA rail stations - use known coordinates for major stations
import numpy as np
from shapely.geometry import Point

septa_stations = gpd.GeoDataFrame({
    'station': [
        '30th Street', 'Suburban Station', 'Jefferson Station',
        'Temple University', 'North Philadelphia', 'Frankford TC',
        'Market-Frankford 69th St', 'Fern Rock TC', 'Girard',
        'Erie', 'Olney TC', 'Allegheny', 'Somerset'
    ],
    'geometry': [
        Point(-75.1822, 39.9566), Point(-75.1641, 39.9531),
        Point(-75.1580, 39.9523), Point(-75.1483, 39.9812),
        Point(-75.1580, 39.9990), Point(-75.0513, 39.9925),
        Point(-75.2607, 39.9614), Point(-75.1197, 40.0612),
        Point(-75.1397, 39.9634), Point(-75.1317, 39.9900),
        Point(-75.1197, 40.0378), Point(-75.1317, 39.9746),
        Point(-75.1317, 39.9862)
    ]
}, crs='EPSG:4326')

print(f"Stations loaded: {len(septa_stations)}")
septa_stations.head()

Stations loaded: 13


,station,geometry
0,30th Street,POINT (-75.1822 39.9566)
1,Suburban Station,POINT (-75.1641 39.9531)
2,Jefferson Station,POINT (-75.158 39.9523)
3,Temple University,POINT (-75.1483 39.9812)
4,North Philadelphia,POINT (-75.158 39.999)


In [23]:
# Step 4b: Calculate distance from each OPA property to nearest SEPTA station
from shapely.ops import nearest_points

septa_stations_proj = septa_stations.to_crs('EPSG:3857')
opa_gdf_proj = opa_gdf.to_crs('EPSG:3857')
septa_union = septa_stations_proj.geometry.unary_union

def dist_to_nearest_septa(geom):
    nearest = nearest_points(geom, septa_union)[1]
    return geom.distance(nearest)

opa_gdf_proj['dist_to_septa_m'] = opa_gdf_proj.geometry.apply(dist_to_nearest_septa)
opa_gdf_proj['dist_to_septa_miles'] = (opa_gdf_proj['dist_to_septa_m'] / 1609.34).round(3)

# Bin into categories
opa_gdf_proj['septa_proximity'] = pd.cut(
    opa_gdf_proj['dist_to_septa_miles'],
    bins=[0, 0.25, 0.5, 1.0, float('inf')],
    labels=['<0.25mi', '0.25-0.5mi', '0.5-1mi', '>1mi']
)

print(opa_gdf_proj['septa_proximity'].value_counts())
opa_gdf_proj[['parcel_number', 'dist_to_septa_miles', 'septa_proximity']].head(10)

C:\Users\zhe\AppData\Local\Temp\ipykernel_65364\3909657941.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  septa_union = septa_stations_proj.geometry.unary_union


septa_proximity
>1mi          481130
0.5-1mi        70878
0.25-0.5mi     23406
<0.25mi         7951
Name: count, dtype: int64


,parcel_number,dist_to_septa_miles,septa_proximity
0,788043500,5.511,>1mi
1,871000023,0.643,0.5-1mi
2,783000004,2.253,>1mi
3,871000071,0.397,0.25-0.5mi
4,132000002,2.104,>1mi
5,871601640,2.136,>1mi
6,874500512,4.671,>1mi
7,871000020,0.626,0.5-1mi
8,871508657,0.570,0.5-1mi
9,871402950,0.461,0.25-0.5mi


In [25]:
# Step 5: Merge all features - fix parcel_number type first
assessments_features['parcel_number'] = assessments_features['parcel_number'].astype(str)
pwd_features['parcel_number'] = pwd_features['parcel_number'].astype(str)
neighborhood_df['parcel_number'] = neighborhood_df['parcel_number'].astype(str)
septa_df['parcel_number'] = septa_df['parcel_number'].astype(str)

summary = (
    assessments_features
    .merge(pwd_features, on='parcel_number', how='left')
    .merge(neighborhood_df, on='parcel_number', how='left')
    .merge(septa_df, on='parcel_number', how='left')
)

print(f"Final feature table rows: {len(summary):,}")
print(f"Columns: {summary.columns.tolist()}")
summary.head()

Final feature table rows: 583,425
Columns: ['parcel_number', 'assessed_value_2023', 'assessed_value_2024', 'assessed_value_2025', 'pct_change_2023_to_2025', 'gross_area', 'lot_area_sqft', 'lot_perimeter', 'lot_shape_ratio', 'neighborhood', 'dist_to_septa_miles', 'septa_proximity']


,parcel_number,assessed_value_2023,assessed_value_2024,assessed_value_2025,pct_change_2023_to_2025,gross_area,lot_area_sqft,lot_perimeter,lot_shape_ratio,neighborhood,dist_to_septa_miles,septa_proximity
0,192321010,100.0,100.0,33800.0,33700.00,<NA>,NaN,NaN,NaN,North Philadelphia/East,0.635,0.5-1mi
1,885181720,400.0,400.0,1200.0,200.00,23,3.679688,9.578474,0.0401,North Philadelphia/East,0.469,0.25-0.5mi
2,182240100,87300.0,87300.0,88800.0,1.72,123,19.625000,28.299282,0.0245,Northern Liberties/Fishtown,0.752,0.5-1mi
3,581128301,12500.0,12500.0,40000.0,220.00,4512,717.230469,126.124331,0.0451,Bustleton,6.598,>1mi
4,331217400,41400.0,41400.0,8800.0,-78.74,<NA>,NaN,NaN,NaN,Kensington,1.852,>1mi


In [26]:
# Step 5: Save model_exploration_summary.md
summary_md = """# Model Exploration Summary

## Data Sources Explored

### 1. OPA Assessments (`core.opa_assessments`)
- 6,890,147 rows of historical assessment records
- Join key: `parcel_number`
- Recommended features:
  - `assessed_value_2023`: market value in 2023
  - `assessed_value_2024`: market value in 2024
  - `assessed_value_2025`: market value in 2025
  - `pct_change_2023_to_2025`: percent change over 2 years (signals appreciation trend)

### 2. PWD Parcels (`core.pwd_parcels`)
- 545,847 parcels with geometry
- Join key: `brt_id` = `parcel_number`
- Recommended features:
  - `lot_area_sqft`: parcel area from geometry (cross-validates OPA total_area)
  - `lot_perimeter`: perimeter of parcel polygon
  - `lot_shape_ratio`: area / perimeter^2 (compactness of lot shape)

### 3. Philadelphia Neighborhoods (OpenDataPhilly)
- 57 neighborhood polygons
- Spatial join to OPA properties via point-in-polygon
- Only 4 out of 583,365 properties had no neighborhood match
- Recommended feature:
  - `neighborhood`: categorical, stronger location signal than zip_code
  - Reduces location categories from ~100 zip codes to 57 neighborhoods

### 4. SEPTA Rail Stations
- 13 major rail/subway stations (MFL, BSL, Regional Rail)
- Recommended features:
  - `dist_to_septa_miles`: continuous distance to nearest station
  - `septa_proximity`: binned category (<0.25mi, 0.25-0.5mi, 0.5-1mi, >1mi)
- Distribution: 81% of properties are >1 mile from a station

## Confirmed Feature List for Model (Issue #11)

| Feature | Source Table | Join Key | Notes |
|---------|-------------|----------|-------|
| `assessed_value_2023` | `core.opa_assessments` | `parcel_number` | Historical value |
| `assessed_value_2024` | `core.opa_assessments` | `parcel_number` | Historical value |
| `assessed_value_2025` | `core.opa_assessments` | `parcel_number` | Most recent assessment |
| `pct_change_2023_to_2025` | `core.opa_assessments` | `parcel_number` | Appreciation signal |
| `lot_area_sqft` | `core.pwd_parcels` | `brt_id` = `parcel_number` | Geometry-derived area |
| `lot_shape_ratio` | `core.pwd_parcels` | `brt_id` = `parcel_number` | Lot compactness |
| `neighborhood` | OpenDataPhilly neighborhoods | spatial join | Categorical location feature |
| `dist_to_septa_miles` | SEPTA stations | spatial join | Transit proximity |
| `septa_proximity` | SEPTA stations | spatial join | Binned transit category |

## Features NOT Recommended
- `zip_code`: collinear with neighborhood, less granular
- Crime data: high null rates, potential bias
- Flood zones: affects <2% of Philadelphia properties
- School catchments: collinear with neighborhood

## Next Steps (Issue #11)
- Use `core.opa_properties` as base table (target: `sale_price`)
- Filter to residential properties (`category_code = 1`)
- Filter to sales since 2015 (`sale_date >= 2015-01-01`)
- Exclude dollar-sales (`sale_price > 1`)
- Join features from the tables above using `parcel_number`
"""

output_path = r"D:\pen\2026\MUSA_5090_Geospatial_Cloud_Computing\group\s26-team5-cama\eda\model_exploration_summary.md"

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(summary_md)

print("model_exploration_summary.md saved!")

model_exploration_summary.md saved!
